In [1]:
import urllib.request

print("Downloading Haar Cascade files...")
urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml', 'haarcascade_frontalface_default.xml')
urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml', 'haarcascade_eye.xml')
print("Download complete.")

Download complete.


In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Update this path if your Windows username or download location is different
train_dir = 'C:/Users/prith/Downloads/archive (1)/data/train'
test_dir = 'C:/Users/prith/Downloads/archive (1)/data/test'

datagen = ImageDataGenerator(rescale=1./255)

print("Loading data...")
train_data = datagen.flow_from_directory(train_dir, target_size=(24, 24), color_mode='grayscale', class_mode='binary', batch_size=32)
test_data = datagen.flow_from_directory(test_dir, target_size=(24, 24), color_mode='grayscale', class_mode='binary', batch_size=32)

print("Class mapping:", train_data.class_indices)

Loading data...
Found 50937 images belonging to 2 classes.
Found 16981 images belonging to 2 classes.
Class mapping: {'awake': 0, 'sleepy': 1}


In [3]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(24, 24, 1)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') 
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Starting training loop...")
model.fit(train_data, validation_data=test_data, epochs=5)

model.save('drowsiness_model.keras')
print("Model saved successfully to your current folder!")

C:\Users\prith\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting training loop...
Epoch 1/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 253s 155ms/step - accuracy: 0.9188 - loss: 0.2114 - val_accuracy: 0.9602 - val_loss: 0.1103
Epoch 2/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 250s 157ms/step - accuracy: 0.9617 - loss: 0.1076 - val_accuracy: 0.9706 - val_loss: 0.0797
Epoch 3/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 246s 154ms/step - accuracy: 0.9687 - loss: 0.0852 - val_accuracy: 0.9754 - val_loss: 0.0652
Epoch 4/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 233s 146ms/step - accuracy: 0.9715 - loss: 0.0774 - val_accuracy: 0.9786 - val_loss: 0.0602
Epoch 5/5
1592/1592 ━━━━━━━━━━━━━━━━━━━━ 259s 144ms/step - accuracy: 0.9755 - loss: 0.0668 - val_accuracy: 0.9799 - val_loss: 0.0563
Model saved successfully to your current folder!


In [5]:
!pip uninstall -y cv2 opencv-python opencv-contrib-python
!pip install opencv-python

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-contrib-python 4.12.0.88
Uninstalling opencv-contrib-python-4.12.0.88:
  Successfully uninstalled opencv-contrib-python-4.12.0.88


You can safely remove it manually.


  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2
import tensorflow as tf

# Load saved model and Haar cascades
model = tf.keras.models.load_model('drowsiness_model.keras')
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

cap = cv2.VideoCapture(0)
closed_frames = 0 

while True:
    ret, frame = cap.read()
    if not ret: break
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 4)
        
        for (ex, ey, ew, eh) in eyes:
            eye = roi_gray[ey:ey+eh, ex:ex+ew]
            eye = cv2.resize(eye, (24, 24))
            eye = eye / 255.0
            eye = eye.reshape(1, 24, 24, 1)
            
            prediction = model.predict(eye, verbose=0)
            
            # 1 = sleepy, 0 = awake
            if prediction > 0.5: 
                closed_frames += 1
                cv2.putText(frame, "Sleepy", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            else: 
                closed_frames = 0
                cv2.putText(frame, "Awake", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
            if closed_frames > 15:
                cv2.putText(frame, "DROWSINESS ALERT!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
                
    cv2.imshow('Drowsiness Detection', frame)
    
    # Press 'q' on the pop-up video window to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cap.release()
cv2.destroyAllWindows()

AttributeError: module 'cv2' has no attribute 'CascadeClassifier'

In [4]:
%pip install --force-reinstall opencv-python

  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 2.7 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/12.6 MB 3.3 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/12.6 MB 3.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.8/12.6 MB 2.6 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.6 MB 2.6 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/12.6 MB 2.6 MB/s eta 0:00:04
   ----------- ---------------------------- 3.7/12.6 MB 2.7 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/12.6 MB 2.7 MB/s eta 0:00:03
   ---------------- ----------------------- 5.2/12.6 MB 2.8 MB/s eta 0:00:03
   ------------------- -------------------- 6.0/12.6 MB 2.8 MB/s eta 0:00:03
   --------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
caer 2.1.1 requires opencv-contrib-python, which is not installed.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import urllib.request
import cv2
import numpy as np
import tensorflow as tf

# 1. Download XML files automatically if they are missing
face_xml = 'haarcascade_frontalface_default.xml'
eye_xml = 'haarcascade_eye.xml'

if not os.path.exists(face_xml):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml', face_xml)

if not os.path.exists(eye_xml):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml', eye_xml)

# 2. Load trained model and face/eye detectors
model = tf.keras.models.load_model('drowsiness_model.keras')
face_cascade = cv2.CascadeClassifier(face_xml)
eye_cascade = cv2.CascadeClassifier(eye_xml)

# 3. Start Webcam
cap = cv2.VideoCapture(0)
closed_frames = 0 

print("Webcam starting... Click the video window and press 'q' to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Unable to access webcam.")
        break
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 4)
        
        for (ex, ey, ew, eh) in eyes:
            eye = roi_gray[ey:ey+eh, ex:ex+ew]
            eye = cv2.resize(eye, (24, 24))
            eye = eye / 255.0
            eye = eye.reshape(1, 24, 24, 1)
            
            prediction = model.predict(eye, verbose=0)
            
            # 1 = Sleepy, 0 = Awake
            if prediction > 0.5: 
                closed_frames += 1
                cv2.putText(frame, "Sleepy", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            else: 
                closed_frames = 0
                cv2.putText(frame, "Awake", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
            if closed_frames > 15:
                cv2.putText(frame, "DROWSINESS ALERT!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
                
    cv2.imshow('Drowsiness Detection', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cap.release()
cv2.destroyAllWindows()

AttributeError: module 'cv2' has no attribute 'CascadeClassifier'

In [2]:
# 1. Uninstall all conflicting packages
%pip uninstall -y cv2 opencv-python opencv-contrib-python opencv-python-headless

# 2. Install the clean release
%pip install opencv-python

# 3. Verify the installation directly in memory
import cv2
print("OpenCV Version:", cv2.__version__)
print("Checking CascadeClassifier support:", hasattr(cv2, 'CascadeClassifier'))

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
Note: you may need to restart the kernel to use updated packages.
OpenCV Version: 5.0.0
Checking CascadeClassifier support: False



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --force-reinstall opencv-python==4.10.0.84

  Using cached numpy-2.5.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
    --------------------------------------- 0.5/38.8 MB 457.3 kB/s eta 0:01:24
    --------------------------------------- 0.5/38.8 MB 457.3 kB/s eta 0:01:24
 

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
caer 2.1.1 requires opencv-contrib-python, which is not installed.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import urllib.request
import cv2
import numpy as np
import tensorflow as tf

# Download required Haar Cascade XML files if missing
face_xml = 'haarcascade_frontalface_default.xml'
eye_xml = 'haarcascade_eye.xml'

if not os.path.exists(face_xml):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml', face_xml)

if not os.path.exists(eye_xml):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml', eye_xml)

# Load trained model and classifiers
model = tf.keras.models.load_model('drowsiness_model.keras')
face_cascade = cv2.CascadeClassifier(face_xml)
eye_cascade = cv2.CascadeClassifier(eye_xml)

cap = cv2.VideoCapture(0)
closed_frames = 0 

print("Webcam starting with eye-tracking bounding boxes. Press 'q' on the window to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to capture video frame.")
        break
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        # Draw a blue rectangle around the entire face
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 4)
        
        for (ex, ey, ew, eh) in eyes:
            eye = roi_gray[ey:ey+eh, ex:ex+ew]
            eye = cv2.resize(eye, (24, 24))
            eye = eye / 255.0
            eye = eye.reshape(1, 24, 24, 1)
            
            prediction = model.predict(eye, verbose=0)
            
            # Prediction logic: 1 = Sleepy, 0 = Awake
            if prediction > 0.5: 
                closed_frames += 1
                status = "Sleepy"
                box_color = (0, 0, 255) # Red for Sleepy
            else: 
                closed_frames = 0
                status = "Awake"
                box_color = (0, 255, 0) # Green for Awake
            
            # Draw bounding box rectangle around the eye region
            cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), box_color, 2)
            
            # Display label directly above the eye box
            cv2.putText(roi_color, status, (ex, ey - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
                
        if closed_frames > 15:
            cv2.putText(frame, "DROWSINESS ALERT!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
                
    cv2.imshow('Drowsiness Detection System', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        
cap.release()
cv2.destroyAllWindows()

Webcam starting with eye-tracking bounding boxes. Press 'q' on the window to quit.


In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. Define test directory path
test_dir = 'C:/Users/prith/Downloads/archive (1)/data/test'

# 2. Load test generator with shuffle=False to lock label ordering
datagen = ImageDataGenerator(rescale=1./255)
test_data_eval = datagen.flow_from_directory(
    test_dir,
    target_size=(24, 24),
    color_mode='grayscale',
    class_mode='binary',
    batch_size=32,
    shuffle=False  # Must be False for exact metric alignment
)

# 3. Load trained model & predict
model = tf.keras.models.load_model('drowsiness_model.keras')
print("Evaluating test dataset predictions...")
raw_predictions = model.predict(test_data_eval)

# Convert predicted probabilities (> 0.5) to binary 0 or 1
y_pred = (raw_predictions > 0.5).astype(int).reshape(-1)
y_true = test_data_eval.classes
class_names = list(test_data_eval.class_indices.keys())

# 4. Compute core metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\n" + "="*45)
print("       OVERALL PERFORMANCE METRICS           ")
print("="*45)
print(f" Accuracy  : {acc * 100:.2f}%")
print(f" Precision : {prec * 100:.2f}%")
print(f" Recall    : {rec * 100:.2f}%")
print(f" F1-Score  : {f1 * 100:.2f}%")
print("="*45)

# 5. Output detailed classification report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# 6. Output Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(f" True Awake  (TN={cm[0][0]}, FP={cm[0][1]})")
print(f" True Sleepy (FN={cm[1][0]}, TP={cm[1][1]})")

Found 16981 images belonging to 2 classes.
Evaluating test dataset predictions...
531/531 ━━━━━━━━━━━━━━━━━━━━ 110s 207ms/step

       OVERALL PERFORMANCE METRICS           
 Accuracy  : 98.09%
 Precision : 98.66%
 Recall    : 97.46%
 F1-Score  : 98.06%

Classification Report:

              precision    recall  f1-score   support

       awake     0.9755    0.9871    0.9813      8591
      sleepy     0.9866    0.9746    0.9806      8390

    accuracy                         0.9809     16981
   macro avg     0.9811    0.9808    0.9809     16981
weighted avg     0.9810    0.9809    0.9809     16981

Confusion Matrix:
 True Awake  (TN=8480, FP=111)
 True Sleepy (FN=213, TP=8177)
